# Quetsal Post-Install Example

This notebook shows the normal user-facing path after `pip install quetsal`: pass `optimization_method="quetsal"` to Qiskit's `generate_preset_pass_manager`.

In [1]:
from qiskit.quantum_info import random_clifford
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins
from IPython.display import Markdown, display

N_QUBITS = 7
SEED = 123
HERON_R2_BASIS = ["cz", "id", "rx", "rz", "rzz", "sx", "x"]
cm = CouplingMap.from_line(N_QUBITS)

In [2]:
def two_qubit_count(circuit):
    return sum(1 for inst in circuit.data if len(inst.qubits) == 2)


def metrics(label, circuit):
    return {
        "label": label,
        "qubits": circuit.num_qubits,
        "ops": len(circuit.data),
        "depth": circuit.depth(),
        "2q_gates": two_qubit_count(circuit),
        "gate_counts": dict(circuit.count_ops()),
    }

In [3]:
raw_circuit = random_clifford(N_QUBITS, seed=SEED).to_circuit()

In [4]:
baseline_pm = generate_preset_pass_manager(
    optimization_level=3,
    basis_gates=HERON_R2_BASIS,
    coupling_map=cm,
    seed_transpiler=SEED,
)
baseline_circuit = baseline_pm.run(raw_circuit)

In [5]:
quetsal_pm = generate_preset_pass_manager(
    optimization_level=1,  # layout + routing + translation
    optimization_method="quetsal",  # Quetsal replaces the optimization stage
    basis_gates=HERON_R2_BASIS,
    coupling_map=cm,
    seed_transpiler=SEED,
)
quetsal_circuit = quetsal_pm.run(raw_circuit)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [6]:
rows = [
    metrics("raw", raw_circuit),
    metrics("qiskit_opt3", baseline_circuit),
    metrics("quetsal", quetsal_circuit),
]

table = [
    "| Circuit | Qubits | Ops | Depth | 2q gates | Gate counts |",
    "|---|---:|---:|---:|---:|---|",
]

for row in rows:
    table.append(
        f"| {row['label']} | {row['qubits']} | {row['ops']} | "
        f"{row['depth']} | {row['2q_gates']} | `{row['gate_counts']}` |"
    )

display(Markdown("\n".join(table)))

| Circuit | Qubits | Ops | Depth | 2q gates | Gate counts |
|---|---:|---:|---:|---:|---|
| raw | 7 | 58 | 35 | 25 | `{'cx': 25, 'h': 15, 's': 15, 'x': 2, 'z': 1}` |
| qiskit_opt3 | 7 | 254 | 156 | 71 | `{'sx': 99, 'cz': 69, 'rz': 57, 'rx': 27, 'rzz': 2}` |
| quetsal | 7 | 237 | 129 | 51 | `{'rz': 127, 'sx': 59, 'cz': 51}` |